In [ ]:
"""
Task 1: HOG feature extraction and visualisation.

Kaggle training images are processed to generate HOG features.
Lecturer-provided testing images are processed separately.

No classifier, model training, validation, or class prediction is used
in Task 1.
"""

import argparse
import csv
import hashlib
from pathlib import Path

import cv2
import numpy as np
from skimage.feature import hog


# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------

TRAIN_DIR = Path(
    r"C:\Users\ACER\Downloads\UCCC2513 MINI PROJECT\Other\asg2\archive"
)

ANNOTATION_FILE = TRAIN_DIR / "annotations.csv"

IMAGE_DIR = TRAIN_DIR / "images"

TEST_DIR = Path(
    r"C:\Users\ACER\Downloads\UCCC2513 MINI PROJECT\Other\asg2\Inputs"
)

IMAGE_SIZE = (64, 64)
DISPLAY_COUNT = 10
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".ppm", ".bmp"}


# ---------------------------------------------------------
# Read and clean annotations
# ---------------------------------------------------------

def read_annotations():
    labels_by_filename = {}

    with ANNOTATION_FILE.open(
        newline="",
        encoding="utf-8-sig"
    ) as file:

        reader = csv.DictReader(file)

        for row in reader:
            filename = row["file_name"].strip()
            category = int(float(row["category"]))
            image_path = IMAGE_DIR / filename

            if not image_path.exists():
                continue

            # Keep only one annotation for each filename
            labels_by_filename.setdefault(filename, category)

    return labels_by_filename


def image_hash(path):
    """Return a SHA-256 hash used to detect exact train-test duplicates."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


# ---------------------------------------------------------
# Automatic traffic-sign region extraction
# ---------------------------------------------------------

def make_colour_mask(image):
    """Create a mask for common red, blue and yellow traffic-sign colours."""

    hsv = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2HSV
    )

    red = cv2.bitwise_or(
        cv2.inRange(
            hsv,
            (0, 70, 40),
            (10, 255, 255)
        ),
        cv2.inRange(
            hsv,
            (170, 70, 40),
            (180, 255, 255)
        )
    )

    blue = cv2.inRange(
        hsv,
        (90, 60, 35),
        (140, 255, 255)
    )

    yellow = cv2.inRange(
        hsv,
        (15, 70, 45),
        (40, 255, 255)
    )

    mask = cv2.bitwise_or(
        cv2.bitwise_or(red, blue),
        yellow
    )

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )

    return cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )


def extract_sign_region(image):
    """Automatically locate and crop the most likely traffic-sign region."""

    mask = make_colour_mask(image)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    for contour in contours:

        area = cv2.contourArea(contour)

        x, y, width, height = cv2.boundingRect(
            contour
        )

        aspect_ratio = width / max(
            height,
            1
        )

        if (
            area >= 80
            and 0.45 <= aspect_ratio <= 1.8
        ):
            candidates.append(
                (
                    area,
                    x,
                    y,
                    width,
                    height
                )
            )

    if not candidates:
        return image

    _, x, y, width, height = max(candidates)

    padding = max(
        3,
        int(0.10 * max(width, height))
    )

    x1 = max(0, x - padding)
    y1 = max(0, y - padding)

    x2 = min(
        image.shape[1],
        x + width + padding
    )

    y2 = min(
        image.shape[0],
        y + height + padding
    )

    cropped_sign = image[y1:y2, x1:x2]

    return (
        cropped_sign
        if cropped_sign.size
        else image
    )


# ---------------------------------------------------------
# Task 1: HOG feature extraction
# ---------------------------------------------------------

def extract_hog(
    image,
    create_visualisation=False
):

    if image is None or image.size == 0:
        raise ValueError(
            "The supplied image is empty."
        )

    # Convert the image to grayscale
    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # Standardise image size
    gray = cv2.resize(
        gray,
        IMAGE_SIZE,
        interpolation=cv2.INTER_AREA
    )

    result = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        visualize=create_visualisation,
        feature_vector=True
    )

    if create_visualisation:

        features, hog_image = result

        hog_image = cv2.normalize(
            hog_image,
            None,
            0,
            255,
            cv2.NORM_MINMAX
        ).astype(np.uint8)

        return (
            features.astype(np.float32),
            gray,
            hog_image
        )

    return result.astype(np.float32)


# ---------------------------------------------------------
# Prepare popup visualisation
# ---------------------------------------------------------

def add_title(image, title):

    panel = cv2.resize(
        image,
        (300, 300)
    )

    title_area = np.zeros(
        (45, panel.shape[1], 3),
        dtype=np.uint8
    )

    cv2.putText(
        title_area,
        title,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    return cv2.vconcat([
        title_area,
        panel
    ])


def create_display(
    image,
    cropped_sign,
    gray,
    hog_image,
    filename,
    category
):

    original_panel = add_title(
        image,
        "Original"
    )

    cropped_panel = add_title(
        cropped_sign,
        "Cropped Sign"
    )

    grayscale_panel = add_title(
        cv2.cvtColor(
            gray,
            cv2.COLOR_GRAY2BGR
        ),
        "Grayscale"
    )

    hog_panel = add_title(
        cv2.cvtColor(
            hog_image,
            cv2.COLOR_GRAY2BGR
        ),
        "HOG Features"
    )

    panels = cv2.hconcat([
        original_panel,
        cropped_panel,
        grayscale_panel,
        hog_panel
    ])

    header = np.zeros(
        (70, panels.shape[1], 3),
        dtype=np.uint8
    )

    cv2.putText(
        header,
        f"File: {filename} | Class: {category:03d}",
        (15, 28),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.70,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    cv2.putText(
        header,
        "Any key = next image | Esc = stop",
        (15, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (180, 220, 255),
        1,
        cv2.LINE_AA
    )

    return cv2.vconcat([
        header,
        panels
    ])


# ---------------------------------------------------------
# Run Task 1 HOG feature extraction
# ---------------------------------------------------------

def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--no-popup",
        action="store_true",
        help="Extract features without opening the visualisation window."
    )

    options = parser.parse_args(args=[])

    if not ANNOTATION_FILE.exists():

        raise FileNotFoundError(
            f"annotations.csv was not found: "
            f"{ANNOTATION_FILE}"
        )

    # -----------------------------------------------------
    # Read annotations
    # -----------------------------------------------------

    labels_by_filename = read_annotations()

    # -----------------------------------------------------
    # Find lecturer testing images
    # -----------------------------------------------------

    test_paths = sorted(
        path
        for path in TEST_DIR.rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )

    if not test_paths:

        raise FileNotFoundError(
            f"No testing images were found: "
            f"{TEST_DIR}"
        )

    # -----------------------------------------------------
    # Remove exact duplicates from training dataset
    # -----------------------------------------------------

    test_hashes = {
        image_hash(path)
        for path in test_paths
    }

    training_images = sorted(
        [
            (IMAGE_DIR / filename, category)
            for filename, category
            in labels_by_filename.items()
            if image_hash(IMAGE_DIR / filename)
            not in test_hashes
        ],
        key=lambda item: item[0].name.lower()
    )

    removed_count = (
        len(labels_by_filename)
        - len(training_images)
    )

    # -----------------------------------------------------
    # Extract HOG features from training images
    # -----------------------------------------------------

    feature_list = []
    label_list = []

    print("=" * 68)
    print("TASK 1: HOG FEATURE EXTRACTION")
    print("=" * 68)

    print(
        f"Unique training images           : "
        f"{len(labels_by_filename)}"
    )

    print(
        f"Exact test-image matches removed : "
        f"{removed_count}"
    )

    print(
        f"Clean training images            : "
        f"{len(training_images)}"
    )

    print(
        f"Lecturer testing images          : "
        f"{len(test_paths)}"
    )

    print("\nExtracting HOG features...")

    for image_path, category in training_images:

        image = cv2.imread(
            str(image_path)
        )

        if image is None:

            print(
                f"Skipped unreadable image: "
                f"{image_path.name}"
            )

            continue

        cropped_sign = extract_sign_region(
            image
        )

        features = extract_hog(
            cropped_sign
        )

        feature_list.append(
            features
        )

        label_list.append(
            category
        )

    if not feature_list:

        raise RuntimeError(
            "No valid training images could be processed."
        )

    # -----------------------------------------------------
    # Create training HOG matrix
    # -----------------------------------------------------

    X_all = np.vstack(
        feature_list
    )

    y_all = np.asarray(
        label_list
    )

    print("\nHOG training extraction completed.")

    print(
        f"Images successfully processed   : "
        f"{len(X_all)}"
    )

    print(
        f"HOG features per image          : "
        f"{X_all.shape[1]}"
    )

    print(
        f"HOG feature matrix shape        : "
        f"{X_all.shape}"
    )

    print(
        f"Label array shape               : "
        f"{y_all.shape}"
    )

    # -----------------------------------------------------
    # Extract HOG features from lecturer test images
    # -----------------------------------------------------

    test_features = []
    test_labels = []
    valid_test_paths = []
    display_results = []

    selected_indices = set(
        np.linspace(
            0,
            len(test_paths) - 1,
            num=min(
                DISPLAY_COUNT,
                len(test_paths)
            ),
            dtype=int
        ).tolist()
    )

    print("\nExtracting lecturer test HOG features...")

    for index, image_path in enumerate(
        test_paths
    ):

        image = cv2.imread(
            str(image_path)
        )

        if image is None:

            print(
                f"Skipped unreadable testing image: "
                f"{image_path.name}"
            )

            continue

        cropped_sign = extract_sign_region(
            image
        )

        try:

            if index in selected_indices:

                features, gray, hog_image = extract_hog(
                    cropped_sign,
                    create_visualisation=True
                )

                category = int(
                    image_path.name.split("_")[0]
                )

                display_results.append(
                    (
                        image,
                        cropped_sign,
                        gray,
                        hog_image,
                        image_path.name,
                        category
                    )
                )

            else:

                features = extract_hog(
                    cropped_sign
                )

            test_features.append(
                features
            )

            test_labels.append(
                int(
                    image_path.name.split("_")[0]
                )
            )

            valid_test_paths.append(
                image_path
            )

        except (cv2.error, ValueError) as error:

            print(
                f"Skipped {image_path.name}: "
                f"{error}"
            )

    if not test_features:

        raise RuntimeError(
            "No valid lecturer testing images could be processed."
        )

    # -----------------------------------------------------
    # Create test HOG matrix
    # -----------------------------------------------------

    X_test = np.vstack(
        test_features
    )

    y_test = np.asarray(
        test_labels
    )

    print("\nLecturer test HOG extraction completed.")

    print(
        f"Images successfully processed   : "
        f"{len(X_test)}"
    )

    print(
        f"HOG features per image          : "
        f"{X_test.shape[1]}"
    )

    print(
        f"HOG feature matrix shape        : "
        f"{X_test.shape}"
    )

    print(
        f"Label array shape               : "
        f"{y_test.shape}"
    )

    # -----------------------------------------------------
    # Save HOG features for Task 2
    # -----------------------------------------------------

    np.save(
        "hog_X_all.npy",
        X_all
    )

    np.save(
        "hog_y_all.npy",
        y_all
    )

    np.save(
        "hog_X_test.npy",
        X_test
    )

    np.save(
        "hog_y_test.npy",
        y_test
    )

    # -----------------------------------------------------
    # Final summary
    # -----------------------------------------------------

    print("\n" + "=" * 68)
    print("TASK 1 COMPLETE")
    print("=" * 68)

    print(
        f"Clean training feature matrix   : "
        f"{X_all.shape}"
    )

    print(
        f"Clean training label array      : "
        f"{y_all.shape}"
    )

    print(
        f"Lecturer test feature matrix    : "
        f"{X_test.shape}"
    )

    print(
        f"Lecturer test label array       : "
        f"{y_test.shape}"
    )

    print("\nSaved files:")

    print("- hog_X_all.npy")
    print("- hog_y_all.npy")
    print("- hog_X_test.npy")
    print("- hog_y_test.npy")

    print("\nClassifier: Not used in Task 1")

    # -----------------------------------------------------
    # Visualisation
    # -----------------------------------------------------

    if options.no_popup:
        return

    if not display_results:
        return

    print("\nPress any key for the next image.")
    print("Press Esc to stop.")

    cv2.namedWindow(
        "Task 1 - HOG Feature Extraction",
        cv2.WINDOW_NORMAL
    )

    for result in display_results:

        display = create_display(
            *result
        )

        cv2.imshow(
            "Task 1 - HOG Feature Extraction",
            display
        )

        key = cv2.waitKey(0) & 0xFF

        if key == 27:
            break

    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# TASK 2: HOG + SVM CLASSIFICATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



# 1. LOAD SAVED HOG FEATURES
print("=" * 70)
print("TASK 2: HOG + SVM CLASSIFICATION")
print("=" * 70)

print("\nLoading HOG features saved from Task 1...")

X_all = np.load("hog_X_all.npy")
y_all = np.load("hog_y_all.npy")

X_test = np.load("hog_X_test.npy")
y_test = np.load("hog_y_test.npy")

print("HOG features loaded successfully.")
print(f"Training feature matrix : {X_all.shape}")
print(f"Training labels         : {y_all.shape}")
print(f"Test feature matrix    : {X_test.shape}")
print(f"Test labels             : {y_test.shape}")



# 2. TRAIN / VALIDATION SPLIT
print("\n" + "=" * 70)
print("TRAIN / VALIDATION SPLIT")
print("=" * 70)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)

print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_validation)}")



# 3. FEATURE SCALING
print("\nScaling HOG features...")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_validation_scaled = scaler.transform(X_validation)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")



# 4. SELECT SVM PARAMETERS, kernel="rbf", c=10, gamma="scale"
print("\n" + "=" * 70)
print("SELECTED SVM PARAMETERS")
print("=" * 70)

kernel = "rbf"
C = 10
gamma = "scale"

print(f"Kernel : {kernel.upper()}")
print(f"C      : {C}")
print(f"Gamma  : {gamma}")



# 5. TRAIN SVM
print("\n" + "=" * 70)
print("SVM TRAINING")
print("=" * 70)

svm_model = SVC(
    kernel=kernel,
    C=C,
    gamma=gamma
)

print("Training SVM...")

svm_model.fit(
    X_train_scaled,
    y_train
)

print("SVM training completed.")



# 6. TRAINING PREDICTION
print("\nMaking training predictions...")

y_train_pred = svm_model.predict(
    X_train_scaled
)

print("Training prediction completed.")



# 7. VALIDATION PREDICTION
print("\nMaking validation predictions...")

y_validation_pred = svm_model.predict(
    X_validation_scaled
)

print("Validation prediction completed.")



# 8. TEST PREDICTION
print("\nMaking test predictions...")

y_test_pred = svm_model.predict(
    X_test_scaled
)

print("Test prediction completed.")



# 9. TRAINING RESULTS
train_accuracy = accuracy_score(
    y_train,
    y_train_pred
)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)



# 10. VALIDATION RESULTS
validation_accuracy = accuracy_score(
    y_validation,
    y_validation_pred
)

validation_precision = precision_score(
    y_validation,
    y_validation_pred,
    average="weighted",
    zero_division=0
)

validation_recall = recall_score(
    y_validation,
    y_validation_pred,
    average="weighted",
    zero_division=0
)

validation_f1 = f1_score(
    y_validation,
    y_validation_pred,
    average="weighted",
    zero_division=0
)



# 11. TEST RESULTS
test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
    zero_division=0
)



# 12. DISPLAY TRAINING AND VALIDATION RESULTS
print("\n" + "=" * 70)
print("TRAINING AND VALIDATION RESULTS")
print("=" * 70)

print("\nTRAINING SET")
print("-" * 70)

print(f"Accuracy  : {train_accuracy:.4f} "
      f"({train_accuracy * 100:.2f}%)")

print(f"Precision : {train_precision:.4f} "
      f"({train_precision * 100:.2f}%)")

print(f"Recall    : {train_recall:.4f} "
      f"({train_recall * 100:.2f}%)")

print(f"F1-score  : {train_f1:.4f} "
      f"({train_f1 * 100:.2f}%)")


print("\nVALIDATION SET")
print("-" * 70)

print(f"Accuracy  : {validation_accuracy:.4f} "
      f"({validation_accuracy * 100:.2f}%)")

print(f"Precision : {validation_precision:.4f} "
      f"({validation_precision * 100:.2f}%)")

print(f"Recall    : {validation_recall:.4f} "
      f"({validation_recall * 100:.2f}%)")

print(f"F1-score  : {validation_f1:.4f} "
      f"({validation_f1 * 100:.2f}%)")


print("\nTEST SET")
print("-" * 70)

print(f"Accuracy  : {test_accuracy:.4f} "
      f"({test_accuracy * 100:.2f}%)")

print(f"Precision : {test_precision:.4f} "
      f"({test_precision * 100:.2f}%)")

print(f"Recall    : {test_recall:.4f} "
      f"({test_recall * 100:.2f}%)")

print(f"F1-score  : {test_f1:.4f} "
      f"({test_f1 * 100:.2f}%)")



# 13. VALIDATION CLASSIFICATION REPORT
print("\n" + "=" * 70)
print("VALIDATION CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_validation,
        y_validation_pred,
        zero_division=0
    )
)



# 14. VALIDATION CONFUSION MATRIX
print("\nGenerating validation confusion matrix...")

cm_validation = confusion_matrix(
    y_validation,
    y_validation_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_validation
)

fig, ax = plt.subplots(figsize=(16, 14))

disp.plot(
    ax=ax,
    xticks_rotation="vertical"
)

ax.set_title(
    f"HOG + SVM - Validation Confusion Matrix\n"
    f"Kernel=RBF, C={C}, Gamma={gamma}"
)

plt.tight_layout()
plt.show()



# 15. FINAL PERFORMANCE COMPARISON
print("\n" + "=" * 70)
print("FINAL PERFORMANCE COMPARISON")
print("=" * 70)

print(f"{'Dataset':<15}"
      f"{'Accuracy':<15}"
      f"{'Precision':<15}"
      f"{'Recall':<15}"
      f"{'F1-score':<15}")

print("-" * 70)

print(f"{'Training':<15}"
      f"{train_accuracy * 100:<15.2f}"
      f"{train_precision * 100:<15.2f}"
      f"{train_recall * 100:<15.2f}"
      f"{train_f1 * 100:<15.2f}")

print(f"{'Validation':<15}"
      f"{validation_accuracy * 100:<15.2f}"
      f"{validation_precision * 100:<15.2f}"
      f"{validation_recall * 100:<15.2f}"
      f"{validation_f1 * 100:<15.2f}")

print(f"{'Test':<15}"
      f"{test_accuracy * 100:<15.2f}"
      f"{test_precision * 100:<15.2f}"
      f"{test_recall * 100:<15.2f}"
      f"{test_f1 * 100:<15.2f}")



# 16. FINAL TEST RESULTS
print("\n" + "=" * 70)
print("FINAL TEST RESULTS - HOG + SVM")
print("=" * 70)

print("Kernel     : RBF")
print("C          : 10")
print("Gamma      : scale")
print(f"Test images: {len(y_test)}")

print()
print(f"Accuracy   : {test_accuracy:.4f} "
      f"({test_accuracy * 100:.2f}%)")

print(f"Precision  : {test_precision:.4f} "
      f"({test_precision * 100:.2f}%)")

print(f"Recall     : {test_recall:.4f} "
      f"({test_recall * 100:.2f}%)")

print(f"F1-score   : {test_f1:.4f} "
      f"({test_f1 * 100:.2f}%)")



# 17. FINAL CLASSIFICATION REPORT
print("\n" + "=" * 70)
print("FINAL TEST CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_test_pred,
        zero_division=0
    )
)



# 18. FINAL CONFUSION MATRIX
print("\nGenerating final confusion matrix...")

cm_test = confusion_matrix(
    y_test,
    y_test_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_test
)

fig, ax = plt.subplots(figsize=(16, 14))

disp.plot(
    ax=ax,
    xticks_rotation="vertical"
)

ax.set_title(
    "HOG + SVM - Final Test Confusion Matrix\n"
    "Kernel=RBF, C=10, Gamma=scale"
)

plt.tight_layout()
plt.show()



# 19. FINAL SUMMARY
print("\n" + "=" * 70)
print("HOG + SVM FINAL SUMMARY")
print("=" * 70)

print("Best Kernel : RBF")
print("Best C      : 10")
print("Best Gamma  : scale")

print("\nTraining Performance")
print(f"Accuracy  : {train_accuracy * 100:.2f}%")
print(f"Precision : {train_precision * 100:.2f}%")
print(f"Recall    : {train_recall * 100:.2f}%")
print(f"F1-score  : {train_f1 * 100:.2f}%")

print("\nValidation Performance")
print(f"Accuracy  : {validation_accuracy * 100:.2f}%")
print(f"Precision : {validation_precision * 100:.2f}%")
print(f"Recall    : {validation_recall * 100:.2f}%")
print(f"F1-score  : {validation_f1 * 100:.2f}%")

print("\nTest Performance")
print(f"Accuracy  : {test_accuracy * 100:.2f}%")
print(f"Precision : {test_precision * 100:.2f}%")
print(f"Recall    : {test_recall * 100:.2f}%")
print(f"F1-score  : {test_f1 * 100:.2f}%")

print("=" * 70)

In [ ]:
# ================================================
# VISUALIZE 10 TRAINING AND 10 TEST RESULTS
# ================================================

import cv2
import matplotlib.pyplot as plt
import numpy as np
import csv
import hashlib
import math

from pathlib import Path
from sklearn.model_selection import train_test_split


# 1. IMAGE HASH FUNCTION

def image_hash(path):
    """Return SHA-256 hash for detecting exact duplicate images."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


# 2. READ TRAINING ANNOTATIONS

labels_by_filename = {}

with ANNOTATION_FILE.open(
    newline="",
    encoding="utf-8-sig"
) as file:

    reader = csv.DictReader(file)

    for row in reader:

        filename = row["file_name"].strip()
        category = int(float(row["category"]))

        image_path = IMAGE_DIR / filename

        if not image_path.exists():
            continue

        labels_by_filename.setdefault(
            filename,
            category
        )


# 3. FIND TEST IMAGES

test_image_files = sorted([
    p
    for p in TEST_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower()
    in [".jpg", ".jpeg", ".png", ".ppm", ".bmp"]
])


print("=" * 60)
print("VISUALIZATION DATA")
print("=" * 60)

print(
    f"Test images found: "
    f"{len(test_image_files)}"
)

print(
    f"Test labels      : "
    f"{len(y_test)}"
)


if len(test_image_files) != len(y_test):

    raise ValueError(
        f"Number of test images ({len(test_image_files)}) "
        f"does not match number of test labels ({len(y_test)})."
    )


# 4. REMOVE TRAIN-TEST DUPLICATES

print(
    "\nChecking for exact train-test duplicates..."
)


test_hashes = {
    image_hash(path)
    for path in test_image_files
}


# Recreate EXACTLY the same training image list
# used in HOG.py

training_images = sorted(
    [
        (IMAGE_DIR / filename, category)
        for filename, category
        in labels_by_filename.items()
        if image_hash(IMAGE_DIR / filename)
        not in test_hashes
    ],
    key=lambda item: item[0].name.lower()
)


print(
    f"Original annotated images : "
    f"{len(labels_by_filename)}"
)

print(
    f"Duplicate images removed  : "
    f"{len(labels_by_filename) - len(training_images)}"
)

print(
    f"Clean training images     : "
    f"{len(training_images)}"
)

print(
    f"HOG training labels       : "
    f"{len(y_all)}"
)


# 5. VERIFY TRAINING IMAGE COUNT

if len(training_images) != len(y_all):

    raise ValueError(
        f"Training image count ({len(training_images)}) "
        f"does not match HOG labels ({len(y_all)})."
    )


# 6. GET TRAINING AND TEST PREDICTIONS

if "y_train_pred" not in globals():

    raise NameError(
        "y_train_pred was not found.\n"
        "Please run Cell 2 first."
    )


if "y_test_pred" not in globals():

    raise NameError(
        "y_test_pred was not found.\n"
        "Please run Cell 2 first."
    )


print(
    f"Training predictions: "
    f"{len(y_train_pred)}"
)

print(
    f"Test predictions    : "
    f"{len(y_test_pred)}"
)


# 7. DISPLAY FUNCTION

def display_results(
    image_files,
    actual_labels,
    predicted_labels,
    title
):

    # Select first 10 images

    indices = np.arange(
        min(10, len(actual_labels))
    )

    fig, axes = plt.subplots(
        2,
        5,
        figsize=(16, 10)
    )

    axes = axes.ravel()


    for i, idx in enumerate(indices):

        # Read image

        image = cv2.imread(
            str(image_files[idx])
        )

        if image is None:

            axes[i].axis("off")

            continue


        # Convert BGR → RGB

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )


        # Display image

        axes[i].imshow(image)


        # Get actual and predicted classes

        actual = int(
            actual_labels[idx]
        )

        predicted = int(
            predicted_labels[idx]
        )


        # Determine result

        if actual == predicted:

            result = "✓ CORRECT"

        else:

            result = "✗ INCORRECT"


        # Display information BELOW image

        axes[i].text(
            0.5,
            -0.12,
            f"Actual: Class {actual}\n"
            f"Predicted: Class {predicted}\n"
            f"{result}",
            transform=axes[i].transAxes,
            ha="center",
            va="top",
            fontsize=10
        )


        axes[i].axis("off")


    # Hide unused spaces

    for i in range(
        len(indices),
        10
    ):

        axes[i].axis("off")


    # Main title

    plt.suptitle(
        title,
        fontsize=16,
        fontweight="bold",
        y=0.98
    )


    # Adjust spacing

    plt.subplots_adjust(
        top=0.90,
        bottom=0.08,
        hspace=0.65,
        wspace=0.20
    )


    plt.show()


# 8. CREATE TRAINING IMAGE PATH LIST

train_image_files = [
    item[0]
    for item in training_images
]


# 9. VERIFY TRAINING LABEL ORDER

train_image_labels = np.array([
    item[1]
    for item in training_images
])


# This should match y_all exactly

if not np.array_equal(
    train_image_labels,
    y_all
):

    raise ValueError(
        "Training image labels do not match y_all.\n"
        "The image ordering may be different from HOG.py."
    )


print(
    "\nTraining image-label order verified ✓"
)


# 10. RECREATE THE SAME TRAINING SPLIT USED IN CELL 2

train_indices, validation_indices = train_test_split(
    np.arange(len(y_all)),
    test_size=0.20,
    random_state=42,
    stratify=y_all
)


train_image_files_split = [
    train_image_files[i]
    for i in train_indices
]


train_labels_split = y_all[
    train_indices
]


# Verify that the recreated training split
# matches the number of predictions

if len(train_image_files_split) != len(y_train_pred):

    raise ValueError(
        f"Training image split ({len(train_image_files_split)}) "
        f"does not match training predictions "
        f"({len(y_train_pred)})."
    )


print(
    f"Training images for visualization: "
    f"{len(train_image_files_split)}"
)


# 11. SHOW 10 TRAINING RESULTS

display_results(
    train_image_files_split,
    train_labels_split,
    y_train_pred,
    "10 Training Results - HOG + SVM"
)


# 12. SHOW 10 TEST RESULTS

display_results(
    test_image_files,
    y_test,
    y_test_pred,
    "10 Test Results - HOG + SVM"
)


# 13. SAVE REMAINING 74 TEST RESULTS AS COMBINED IMAGES

# 10 images per page (2 rows × 5 columns)

# Create output folder

output_dir = Path(
    "test_result_images_combined"
)

output_dir.mkdir(
    exist_ok=True
)


# Remaining test images:
# image 11 to image 84

remaining_indices = list(
    range(
        10,
        len(test_image_files)
    )
)


# 10 images per page

images_per_page = 10

rows = 2
cols = 5


# Calculate number of pages

num_pages = math.ceil(
    len(remaining_indices)
    / images_per_page
)


print(
    "\n" + "=" * 70
)

print(
    "GENERATING COMBINED TEST RESULT IMAGES"
)

print(
    "=" * 70
)

print(
    f"Remaining test images : "
    f"{len(remaining_indices)}"
)

print(
    f"Images per page       : "
    f"{images_per_page}"
)

print(
    f"Number of pages       : "
    f"{num_pages}"
)

print(
    "=" * 70
)


for page in range(num_pages):


    # Get indices for this page

    start = (
        page
        * images_per_page
    )

    end = min(
        start + images_per_page,
        len(remaining_indices)
    )


    page_indices = remaining_indices[
        start:end
    ]


    # Create combined figure

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(15, 7),
        facecolor="white"
    )


    axes = axes.flatten()


    # Add images to the page

    for position, idx in enumerate(
        page_indices
    ):


        # Read original image

        image = cv2.imread(
            str(test_image_files[idx])
        )


        if image is None:

            axes[position].axis(
                "off"
            )

            continue


        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )


        actual = int(
            y_test[idx]
        )

        predicted = int(
            y_test_pred[idx]
        )


        if actual == predicted:

            result = "✓ CORRECT"

        else:

            result = "✗ INCORRECT"


        # Display image

        axes[position].imshow(
            image
        )

        axes[position].axis(
            "off"
        )


        # Add result information

        axes[position].set_title(
            f"Actual: Class {actual}\n"
            f"Predicted: Class {predicted}\n"
            f"{result}",
            fontsize=9,
            fontweight="bold"
        )


    # Hide unused spaces on last page

    for position in range(
        len(page_indices),
        len(axes)
    ):

        axes[position].axis(
            "off"
        )


    # Adjust spacing

    plt.tight_layout()


    # Save combined page

    output_path = (
        output_dir
        / f"main_hogsvm_test_results_page_{page + 1:02d}.jpg"
    )


    fig.savefig(
        output_path,
        dpi=200,
        facecolor="white",
        bbox_inches="tight"
    )


    plt.close(fig)


    print(
        f"Saved: {output_path}"
    )


# 14. CONFIRM SAVED FILES

saved_files = sorted(
    output_dir.glob("*.jpg")
)


print(
    "\n" + "=" * 70
)

print(
    "COMBINED TEST RESULTS SAVED"
)

print(
    "=" * 70
)

print(
    f"Output folder : "
    f"{output_dir.resolve()}"
)

print(
    f"Pages saved   : "
    f"{len(saved_files)}"
)

print(
    "=" * 70
)